# AIFF embed new image cover 

In [1]:
# -----######-----###### CORE IMPORTABLE FUNCTION (AIFF Cover Replace • ID3 APIC Only) -----######-----###### #

import os
import pandas as pd
from tqdm import tqdm
from mutagen.aiff import AIFF
from mutagen.id3 import APIC, ID3


def _coverart_0603_replace_aiff_GET_df_status(
        root_folder,
        png_cover_path,
        export_csv_path=None
):
    """
    Replace embedded cover art for all AIFF files inside a folder.

    • Only replaces APIC (cover art)
    • Keeps ALL metadata
    • Keeps audio untouched
    • Works recursively

    Returns:
        DataFrame log
    """

    with open(png_cover_path, "rb") as f:
        img_data = f.read()

    results = []

    aiff_paths = []

    for root, _, files in os.walk(root_folder):
        for file in files:

            if file.startswith("._") or file.startswith(".DS"):
                continue

            if file.lower().endswith((".aiff", ".aif")):
                aiff_paths.append(os.path.join(root, file))

    pbar = tqdm(aiff_paths, desc="TQM | AIFF Cover Replace", unit="file")

    for path in pbar:

        status = "fail"
        error = None

        try:

            audio = AIFF(path)

            if audio.tags is None:
                audio.add_tags()

            tags = audio.tags

            # remove existing covers
            tags.delall("APIC")

            # add new cover
            tags.add(
                APIC(
                    encoding=3,
                    mime="image/png",
                    type=3,
                    desc="Cover",
                    data=img_data
                )
            )

            audio.save()

            status = "ok"

        except Exception as e:
            error = str(e)

        results.append(
            {
                "Path": path,
                "status": status,
                "error": error
            }
        )

    df_log = pd.DataFrame(results)

    if export_csv_path:
        df_log.to_csv(export_csv_path, index=False)

    return df_log

In [2]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

root_folder = "/Users/yerik/Desktop/THIS"
png_cover = "/Users/yerik/Desktop/THIS/___ashton.png"

df_log = _coverart_0603_replace_aiff_GET_df_status(
    root_folder,
    png_cover
)

TQM | AIFF Cover Replace: 100%|██████████████████████████████████████████| 172/172 [00:00<00:00, 406.19file/s]
